# Sudoku RL Training on Kaggle

This notebook contains the code to train a reinforcement learning model (MaskablePPO) to solve Sudoku puzzles using `stable-baselines3` and `sb3-contrib`.

### Prerequisites:
1. Ensure you have a GPU enabled in the Kaggle notebook settings (P100 or T4 x2).
2. Add the [Sudoku](https://www.kaggle.com/datasets/bryanpark/sudoku) dataset to your notebook input.

In [23]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/rohanrao/sudoku/sudoku.csv


In [24]:
!pip install stable-baselines3[extra] sb3-contrib gymnasium shimmy

In [25]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import matplotlib
matplotlib.use('Agg') # Use Agg for headless environment
import matplotlib.pyplot as plt
import os
import pandas as pd
from stable_baselines3 import PPO, DQN
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import BaseCallback, EvalCallback
from stable_baselines3.common.monitor import Monitor
from sb3_contrib.common.wrappers import ActionMasker
from sb3_contrib import MaskablePPO
import warnings
warnings.filterwarnings('ignore')

# Ensure directories exist for output
os.makedirs("renders", exist_ok=True)
os.makedirs("models",  exist_ok=True)

In [28]:
class SudokuEnv(gym.Env):
    def __init__(self, df=None):
        super().__init__()
        self.df = df
        self.use_dataset = df is not None
        self.default_puzzle = np.array([
            [5,3,0, 0,7,0, 0,0,0], [6,0,0, 1,9,5, 0,0,0], [0,9,8, 0,0,0, 0,6,0],
            [8,0,0, 0,6,0, 0,0,3], [4,0,0, 8,0,3, 0,0,1], [7,0,0, 0,2,0, 0,0,6],
            [0,6,0, 0,0,0, 2,8,0], [0,0,0, 4,1,9, 0,0,5], [0,0,0, 0,8,0, 0,7,9],
        ])
        self.solution = np.zeros((9, 9))
        self.observation_space = spaces.Box(low=0.0, high=1.0, shape=(9, 9, 9), dtype=np.float32)
        self.action_space = spaces.Discrete(2 * 9 * 9 * 9)
        self.board = None; self.candidates = None; self.given_mask = None; self.failed_actions = set()

    def _decode_action(self, action):
        level = action // 729; rem = action % 729
        row = rem // 81; rem = rem % 81; col = rem // 9; dig = rem % 9
        return level, row, col, dig

    def _is_valid_candidate(self, row, col, digit_1indexed):
        d = digit_1indexed
        if d in self.board[row, :]: return False
        if d in self.board[:, col]: return False
        br, bc = (row // 3) * 3, (col // 3) * 3
        if d in self.board[br:br+3, bc:bc+3]: return False
        return True

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        if self.use_dataset:
            idx = np.random.randint(len(self.df))
            self.puzzle = np.array(list(self.df.iloc[idx]['puzzle']), dtype=int).reshape(9, 9)
            self.solution = np.array(list(self.df.iloc[idx]['solution']), dtype=int).reshape(9, 9)
        else:
            self.puzzle = self.default_puzzle
        self.board = self.puzzle.copy().astype(int); self.given_mask = (self.puzzle > 0)
        self.total_reward = 0.0; self.failed_actions.clear()
        self.candidates = np.zeros((9, 9, 9), dtype=bool)
        for i in range(9): 
            for j in range(9):
                if self.given_mask[i, j]: self.candidates[i, j, self.puzzle[i, j]-1] = True
        return self.candidates.astype(np.float32), {}

    def step(self, action):
        action = int(action); level, row, col, dig = self._decode_action(action)
        digit_1indexed = dig + 1; reward = 0.0; info = {}
        if self.given_mask[row, col]: reward = -0.3
        elif level == 0:
            if self.candidates[row, col, dig]: reward = -0.1
            elif self._is_valid_candidate(row, col, digit_1indexed): self.candidates[row, col, dig] = True; reward = 0.5
            else: reward = -0.5
        elif level == 1:
            if self.board[row, col] != 0: reward = -0.3
            elif not self.candidates[row, col, dig]: reward = -0.5
            elif digit_1indexed == self.solution[row, col]:
                self.board[row, col] = digit_1indexed; self.candidates[row, col, :] = False; self.candidates[row, col, dig] = True; reward = 1.0
            else: reward = -1.0
        if reward < 0: self.failed_actions.add(action)
        self.total_reward += reward; done = all(self.board[i,j] != 0 for i in range(9) for j in range(9) if not self.given_mask[i,j])
        if done: reward += 10.0; self.total_reward += 10.0
        return self.candidates.astype(np.float32), reward, done, False, info

    def action_masks(self):
        mask = np.zeros(1458, dtype=bool)
        for act in range(1458):
            if act in self.failed_actions: continue
            level, row, col, dig = self._decode_action(act)
            if self.given_mask[row, col]: continue
            if level == 0 and not self.candidates[row, col, dig] and self._is_valid_candidate(row, col, dig+1): mask[act] = True
            elif level == 1 and self.board[row, col] == 0 and self.candidates[row, col, dig]: mask[act] = True
        return mask

    def render(self, **kwargs): pass

In [29]:
def find_csv():
    for root, dirs, files in os.walk("/kaggle/input"): 
        for f in files: 
            if f.endswith("sudoku.csv"): return os.path.join(root, f)
    return "sudoku/sudoku.csv" if os.path.exists("sudoku/sudoku.csv") else None

csv_path = find_csv()
if csv_path:
    print(f"✅ Found dataset at {csv_path}")
    df = pd.read_csv(csv_path)
    train_env = make_vec_env(lambda: Monitor(ActionMasker(SudokuEnv(df=df), lambda e: e.action_masks())), n_envs=4)
    eval_env = ActionMasker(SudokuEnv(df=df), lambda e: e.action_masks())
    
    model = MaskablePPO("MlpPolicy", train_env, verbose=0, learning_rate=3e-4)
    print("🚀 Starting Training (Stability Fix Applied)...")
    model.learn(total_timesteps=1_000_000, callback=SudokuCallback(eval_env), progress_bar=False)
    model.save("models/final_sudoku_model")
    print("\n✅ Training complete!")
else:
    print("❌ Dataset not found!")

✅ Found dataset at /kaggle/input/datasets/rohanrao/sudoku/sudoku.csv
🚀 Starting Training (Stability Fix Applied)...
   [Training] Step 1,000...
   [Training] Step 2,000...
   [Training] Step 3,000...
   [Training] Step 4,000...
   [Training] Step 5,000...
   [Training] Step 6,000...
   [Training] Step 7,000...
   [Training] Step 8,000...
   [Training] Step 9,000...
   [Training] Step 10,000...

📊 Step 10,000 | Avg Reward: 66.6 | Solve Rate: 100%
   💾 New best model saved!
   [Training] Step 11,000...
   [Training] Step 12,000...
   [Training] Step 13,000...
   [Training] Step 14,000...
   [Training] Step 15,000...
   [Training] Step 16,000...
   [Training] Step 17,000...
   [Training] Step 18,000...
   [Training] Step 19,000...
   [Training] Step 20,000...

📊 Step 20,000 | Avg Reward: 64.7 | Solve Rate: 100%
   [Training] Step 21,000...
   [Training] Step 22,000...
   [Training] Step 23,000...
   [Training] Step 24,000...
   [Training] Step 25,000...
   [Training] Step 26,000...
   [Tr

KeyboardInterrupt: 

In [ ]:
if os.path.exists("models/final_sudoku_model.zip"):
    print("🧪 Testing Final Model...")
    model = MaskablePPO.load("models/final_sudoku_model")
    test_env = ActionMasker(SudokuEnv(df=df), lambda e: e.action_masks())
    solved = 0; n_test = 50
    for i in range(n_test):
        obs, _ = test_env.reset()
        for _ in range(1000):
            a, _ = model.predict(obs, action_masks=test_env.action_masks(), deterministic=True)
            obs, r, d, _, _ = test_env.step(a)
            if d: solved += 1; break
    print(f"\n🏆 Final Solve Rate: {solved}/{n_test} = {solved/n_test*100:.0f}%")